# 🌿 Precision AgriTech Platform - Prompt-to-Prompt Asset Generator (v2.1)

**Paper Reference:** *Prompt-to-Prompt Image Editing with Cross Attention Control* (Hertz et al., 2022 - arXiv:2208.01626)
**Target Feature:** Feature #1 / TODO-107 (Interactive 0–100% Disease Progression Slider)

This notebook generates photorealistic, geometrically aligned leaf images across 3 disease progression stages (`Stage 0: Healthy` $\to$ `Stage 1: Early Lesions` $\to$ `Stage 2: Severe Necrosis`) for **Tomato**, **Potato**, **Corn**, and **Apple**.

### Key Improvements in v2.1:
- **Classifier-Free Guidance (CFG) Isolation:** Separates unconditional and conditional attention streams, ensuring structural leaf preservation.
- **Safety Checker Bypass:** Disables HuggingFace built-in safety checker (`safety_checker=None`), which falsely classifies dark brown necrotic lesions, rot spots, and rust pustules as NSFW gore and blanks them into black images.
- **Auto-Resume:** Re-running skips already-generated valid crops, so you don't lose previous progress.
- **Automated Sanity Check:** Verifies every plate has non-zero pixels before export.

> **Runtime Requirement:** Ensure GPU is selected (`Runtime` $\to$ `Change runtime type` $\to$ `T4 GPU`).

In [ ]:
# Step 1: Install required libraries
!pip install -q diffusers transformers accelerate torch torchvision pillow numpy

In [ ]:
# Step 2: Define CFG-Aware Cross-Attention Control & UNet Hooking
import torch
import numpy as np
from typing import List, Union, Tuple
from PIL import Image, ImageOps, ImageChops, ImageFilter
from diffusers import StableDiffusionPipeline, DDIMScheduler

class AttentionControlEdit:
    def __init__(self, prompts: List[str], num_steps: int, cross_replace_steps: float = 0.75, self_replace_steps: float = 0.60):
        self.cur_step = 0
        self.num_steps = num_steps
        self.batch_size = len(prompts)
        self.cross_replace_steps = (0.0, cross_replace_steps) if isinstance(cross_replace_steps, float) else cross_replace_steps
        self.self_replace_steps = self_replace_steps

    def step(self):
        self.cur_step += 1

    def __call__(self, attn: torch.Tensor, is_cross: bool, place_in_unet: str) -> torch.Tensor:
        total_heads, pixels, tokens = attn.shape
        b_prompts = self.batch_size

        # Stable Diffusion with CFG doubles batch into [unconditional, conditional]
        is_cfg = (total_heads % (2 * b_prompts) == 0)
        num_streams = 2 if is_cfg else 1
        heads_per_item = total_heads // (num_streams * b_prompts)

        if self.cur_step < self.num_steps * self.cross_replace_steps[1]:
            if is_cross:
                # Explicitly isolate [stream, prompt_idx, head, pixels, tokens]
                reshaped = attn.view(num_streams, b_prompts, heads_per_item, pixels, tokens)
                for s in range(num_streams):
                    source_attn = reshaped[s, 0].clone()
                    for b in range(1, b_prompts):
                        reshaped[s, b] = source_attn
                attn = reshaped.view(total_heads, pixels, tokens)
            else:
                if self.cur_step < self.num_steps * self.self_replace_steps:
                    reshaped = attn.view(num_streams, b_prompts, heads_per_item, pixels, tokens)
                    for s in range(num_streams):
                        source_attn = reshaped[s, 0].clone()
                        for b in range(1, b_prompts):
                            reshaped[s, b] = source_attn
                    attn = reshaped.view(total_heads, pixels, tokens)

        return attn

def register_attention_control(model, controller):
    attn_modules = [m for m in model.unet.modules() if m.__class__.__name__ == 'Attention']
    last_module = attn_modules[-1] if attn_modules else None

    class P2PAttentionProcessor:
        def __init__(self, place, is_last=False):
            self.place = place
            self.is_last = is_last

        def __call__(self, attn_module, hidden_states, encoder_hidden_states=None, **kwargs):
            b_size, _, _ = hidden_states.shape
            is_cross = encoder_hidden_states is not None
            enc_states = encoder_hidden_states if is_cross else hidden_states

            q = attn_module.to_q(hidden_states)
            k = attn_module.to_k(enc_states)
            v = attn_module.to_v(enc_states)

            inner_dim = k.shape[-1]
            head_dim = inner_dim // attn_module.heads

            q = q.view(b_size, -1, attn_module.heads, head_dim).transpose(1, 2)
            k = k.view(b_size, -1, attn_module.heads, head_dim).transpose(1, 2)
            v = v.view(b_size, -1, attn_module.heads, head_dim).transpose(1, 2)

            scores = torch.matmul(q, k.transpose(-1, -2)) * (1.0 / (head_dim ** 0.5))
            probs = scores.softmax(dim=-1)
            probs_flat = probs.view(b_size * attn_module.heads, q.shape[2], k.shape[2])

            probs_flat = controller(probs_flat, is_cross, self.place)
            probs = probs_flat.view(b_size, attn_module.heads, q.shape[2], k.shape[2])

            out = torch.matmul(probs, v).transpose(1, 2).reshape(b_size, -1, attn_module.heads * head_dim)
            out = attn_module.to_out[1](attn_module.to_out[0](out))

            if self.is_last:
                controller.step()

            return out

    for name, module in model.unet.named_modules():
        if module.__class__.__name__ == 'Attention':
            place = 'mid' if 'mid_block' in name else ('down' if 'down_blocks' in name else 'up')
            is_last = (module is last_module)
            module.set_processor(P2PAttentionProcessor(place, is_last=is_last))

print('CFG-Aware P2P Core functions defined successfully!')

In [ ]:
# Step 3: Normalization & WebGL Mask Extraction Utilities
def remove_white_background(img: Image.Image, threshold: int = 240) -> Image.Image:
    img = img.convert('RGBA')
    data = np.array(img)
    r, g, b = data[:, :, 0], data[:, :, 1], data[:, :, 2]
    white_mask = (r > threshold) & (g > threshold) & (b > threshold)
    data[white_mask, 3] = 0
    return Image.fromarray(data)

def compute_difference_mask(img_base: Image.Image, img_target: Image.Image) -> Image.Image:
    diff = ImageChops.difference(img_base.convert('L'), img_target.convert('L'))
    diff = ImageOps.autocontrast(diff, cutoff=2)
    return diff.filter(ImageFilter.GaussianBlur(radius=3.0))

def normalize_to_target_spec(img: Image.Image, target_size=(800, 800)) -> Image.Image:
    img = img.convert('RGBA')
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    final_img = Image.new('RGBA', target_size, (0, 0, 0, 0))
    paste_x = (target_size[0] - img.width) // 2
    paste_y = (target_size[1] - img.height) // 2
    final_img.paste(img, (paste_x, paste_y), img)
    return final_img

print('Post-processing utilities ready!')

In [ ]:
# Step 4: Load Stable Diffusion v1.5 Model & Generate Assets for all 4 Crops
import os
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

scheduler = DDIMScheduler(
    beta_start=0.00085, beta_end=0.012, beta_schedule='scaled_linear',
    clip_sample=False, set_alpha_to_one=False
)

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    scheduler=scheduler,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)

CROP_CONFIGS = {
    'tomato': {
        'seed': 42,
        'cross_replace': 0.70,
        'prompts': [
            'macro studio photograph of a single healthy green tomato leaflet with distinct venation, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
            'macro studio photograph of a single green tomato leaflet with small dark brown concentric target-like lesions and faint yellow chlorotic halos, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
            'macro studio photograph of a single withered tomato leaflet with extensive dark brown necrotic patches, severe dry tissue collapse, and leaf margin yellowing, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
        ]
    },
    'potato': {
        'seed': 108,
        'cross_replace': 0.70,
        'prompts': [
            'macro studio photograph of a single broad oval healthy green potato leaflet, prominent center vein, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
            'macro studio photograph of a single green potato leaflet with irregular water-soaked dark brown-black lesions and pale green margins, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
            'macro studio photograph of a single decaying potato leaflet with large dark brown necrotic blotches, shriveled leaf edges, and severe tissue rot, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
        ]
    },
    'corn': {
        'seed': 204,
        'cross_replace': 0.70,
        'prompts': [
            'macro studio photograph of a single long lanceolate healthy green corn leaf blade with parallel veins, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
            'macro studio photograph of a single green corn leaf blade with scattered elongated cinnamon-brown powdery pustules, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
            'macro studio photograph of a single dry corn leaf blade with dense confluent dark brown rust pustules, necrotic scorch streaks, and tissue brittleness, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
        ]
    },
    'apple': {
        'seed': 312,
        'cross_replace': 0.70,
        'prompts': [
            'macro studio photograph of a single elliptical healthy green apple leaf with finely serrated margins, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
            'macro studio photograph of a single green apple leaf with discrete dull olive-green to black velvety spots, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
            'macro studio photograph of a single distorted apple leaf with large corky brown scab lesions, puckered lamina, and severe necrotic leaf margins, centered, flat lay, isolated on a clean pure white background, sharp focus, 8k',
        ]
    }
}

out_dir = Path('progression_output')
out_dir.mkdir(exist_ok=True)

for crop_id, cfg in CROP_CONFIGS.items():
    crop_dir = out_dir / crop_id
    crop_dir.mkdir(exist_ok=True)

    # Check if already successfully generated to allow seamless resume
    expected_files = [
        crop_dir / f'{crop_id}_stage0_healthy.webp',
        crop_dir / f'{crop_id}_stage1_early.webp',
        crop_dir / f'{crop_id}_stage2_severe.webp',
        crop_dir / f'{crop_id}_stage1_mask.webp',
        crop_dir / f'{crop_id}_stage2_mask.webp',
    ]
    if all(p.exists() and p.stat().st_size > 2000 for p in expected_files):
        print(f'[✓] {crop_id} already generated with valid plates. Skipping to save GPU time...')
        continue

    print(f'\n[+] Generating assets for: {crop_id}...')

    controller = AttentionControlEdit(
        prompts=cfg['prompts'],
        num_steps=50,
        cross_replace_steps=cfg['cross_replace'],
        self_replace_steps=0.55,
    )
    register_attention_control(pipe, controller)

    generator = torch.Generator(device=device).manual_seed(cfg['seed'])
    with torch.autocast('cuda' if device == 'cuda' else 'cpu'):
        res = pipe(cfg['prompts'], num_inference_steps=50, guidance_scale=7.5, generator=generator).images

    # Sanity check: Ensure generated images are not all-black
    for idx, raw_img in enumerate(res):
        arr = np.array(raw_img)
        if np.all(arr == 0):
            raise RuntimeError(f'Error: Generated image {idx} for {crop_id} is completely black!')

    s0 = normalize_to_target_spec(remove_white_background(res[0]))
    s1 = normalize_to_target_spec(remove_white_background(res[1]))
    s2 = normalize_to_target_spec(remove_white_background(res[2]))
    m1 = compute_difference_mask(s0, s1)
    m2 = compute_difference_mask(s1, s2)

    s0.save(crop_dir / f'{crop_id}_stage0_healthy.webp', 'WEBP', quality=90)
    s1.save(crop_dir / f'{crop_id}_stage1_early.webp', 'WEBP', quality=90)
    s2.save(crop_dir / f'{crop_id}_stage2_severe.webp', 'WEBP', quality=90)
    m1.save(crop_dir / f'{crop_id}_stage1_mask.webp', 'WEBP', quality=85)
    m2.save(crop_dir / f'{crop_id}_stage2_mask.webp', 'WEBP', quality=85)
    print(f'[✓] Completed {crop_id} with valid non-zero leaf plates!')

# Zip results for easy download
!zip -r p2p_progression_assets.zip progression_output
from google.colab import files
files.download('p2p_progression_assets.zip')
print('Done! Clean asset ZIP downloaded successfully!')